# Задача A — Адаптивный решатель дискретных головоломок

**Yandex ML Cup — трек ML**

**Цель:** один универсальный алгоритм, который за **50 мин на обучение** и **25 мин на решение** решает *любую* обратимую дискретную головоломку через единый API `gym.py` — включая скрытые головоломки, которых он никогда не видел.

**Итоговый скор: 82** (бейзлайн начинался с ~63).

---

## Головоломки

| Головоломка | Тип хода | Механика |
|---|---|---|
| `game_15_2d` — пятнашки | `SWAP` + `EMPTY` | двигаем плитки в пустую клетку |
| `toggle_lights` — «Лампочки» | `TOGGLE` | нажатие клетки переключает её строку и столбец |
| `cylinder_game` — цилиндр Варикон | `ROTATE` | поворот колец цветного цилиндра |
| *скрытые* | неизвестно | тот же API, неизвестная механика |

Решатель общается со средой только через `reset`, `valid_actions`, `step`, `is_solved`, `encode_state`. Ему **никогда не сообщают, в какую головоломку он играет** — именно поэтому он «адаптивный».

## Основная идея — обучаем функцию ценности `V(s)`

Бейзлайн обучает маленькую нейросеть `V(s)`, которая оценивает, **сколько ходов осталось** до решённого состояния, и использует её как эвристику в поиске A* (`f = g + V`).

**Откуда берутся обучающие данные — обратные блуждания.**  
Стартуем из *решённого* состояния и делаем случайные допустимые ходы. После `k` шагов состояние находится (не более чем) в `k` ходах от решения. Бесплатная размеченная выборка, без человеческих решений:

```
solved ──ход──▶ s1 (расст.≈1) ──ход──▶ s2 (расст.≈2) ──▶ ... ──▶ sk (расст.≈k)
```

**Сеть не зависит от конкретной головоломки.** Каждая клетка превращается в вектор из 15 чисел (3D-позиция, тип/значение содержимого, тип/значение цели, флаги совпадения). Архитектура: MLP по токенам → mean+max pooling → маленькая голова → `softplus`. Поскольку сеть читает через `encode_state`, *одни и те же веса* работают на любой головоломке независимо от размера поля.

## Инсайт №1 — обученная `V` полезна, но её всегда недостаточно

Главный вывод из локальных экспериментов: **точные, учитывающие структуру решатели побеждают обученный поиск везде, где они применимы.** ML-сеть ценности — это *запасной вариант*, а не главный герой.

Самый наглядный пример — `toggle_lights`.

## Инсайт №2 — «Лампочки» это линейная алгебра над GF(2)

Каждая лампочка либо горит (1), либо нет (0). Нажатие клетки переключает фиксированный набор лампочек, а нажать её **дважды — то же самое, что не нажимать** (mod 2). Порядок не важен. Значит, вся головоломка — это система линейных уравнений над полем из двух элементов, **GF(2)**:

$$A \mathbf{x} = \mathbf{b} \pmod 2$$

- `A` — какие лампочки переключает каждая кнопка
- `b` — текущее состояние (что горит)
- `x` — какие кнопки нажать (решение)

Решаем методом Гаусса по модулю 2. Это **точно и мгновенно** — нашло оптимальное решение на **20/20** инстансах (скор 1.52) там, где beam search и A* дали **0.0**.

In [ ]:
import numpy as np

def solve_gf2(A, b):
    """Gaussian elimination over GF(2). Returns x such that A @ x == b (mod 2)."""
    A = A.copy() % 2
    b = b.copy() % 2
    n_rows, n_cols = A.shape
    pivot_col_of_row = []
    row = 0
    for col in range(n_cols):
        # find a row at/below `row` with a 1 in this column
        piv = None
        for r in range(row, n_rows):
            if A[r, col]:
                piv = r
                break
        if piv is None:
            continue
        A[[row, piv]] = A[[piv, row]]          # swap into place
        b[[row, piv]] = b[[piv, row]]
        for r in range(n_rows):                 # eliminate the 1s elsewhere
            if r != row and A[r, col]:
                A[r] ^= A[row]                  # XOR == subtraction mod 2
                b[r] ^= b[row]
        pivot_col_of_row.append(col)
        row += 1

    x = np.zeros(n_cols, dtype=np.int8)
    for r, col in enumerate(pivot_col_of_row):
        x[col] = b[r]
    return x

# Tiny demo: 3 buttons, button i toggles lights i and i+1
A = np.array([[1,1,0],
              [0,1,1],
              [0,0,1]], dtype=np.int8)
b = np.array([1,0,1], dtype=np.int8)   # current lit state
x = solve_gf2(A, b)
print('press buttons:', x)
print('check A@x % 2 == b :', np.array_equal((A @ x) % 2, b))

## Инсайт №3 — beam search лучший *универсальный* запасной вариант

Для «скользящих» и неизвестных головоломок **beam search** с мягкой оценкой *несовпадения* (число клеток не на своих местах) обошёл всё остальное, что обобщается:

| Головоломка | Метод | Скор | Решено |
|---|---|---|---|
| `game_15_2d` | beam + оценка Manhattan/LC | 0.0–0.19 | хрупко, избегаем |
| `game_15_2d` | **beam + оценка несовпадения** | **0.51** | 4/8 |
| `game_15_2d` | быстрый массивный IDA* (EMPTY+SWAP) | 1.21 | 6/8 |
| `game_15_2d` | + резерв на повтор + динамический запас | **1.39** | 7/8 |
| `cylinder_game` | greedy | 1.07 | 5/8 |
| `cylinder_game` | **длинный beam перед A*** | **1.75** | 7/8 |

Контринтуитивный вывод: **оценка Manhattan + linear-conflict *вредила* beam** — она схлопывала разнообразие, и beam застревал. Более мягкая оценка несовпадения сохраняла разнообразие фронта и решала больше.

## Каскад решателей

Итоговый `solve.py` для каждого инстанса запускает **каскад с учётом структуры** — сначала самое дешёвое/точное, обобщённый поиск в конце:

```
определяем структуру ходов через valid_actions / encode_state
         │
  чистый TOGGLE?      ── да ─▶  точный решатель GF(2)          (оптимально, мгновенно)
         │ нет
  EMPTY + SWAP?       ── да ─▶  быстрый массивный IDA* (Manhattan+LC)
         │                        └─ остаток бюджета ─▶ beam (несовпадение)
         │ нет  (вращательные / скрытые)
         └────────────────────▶  длинный beam ─▶ A* с обученной V  (запасной вариант)
```

Таблица обратного BFS (построенная из решённого состояния во время обучения) даёт оптимальные концовки для любого инстанса, который она покрывает.

## Баги, которые реально подняли скор

Большая часть прироста с 63 → 82 пришла от **исправления багов**, а не от новых моделей:

| Исправление | Скор |
|---|---|
| Старт (бейзлайн: beam + простой RL-ранжировщик) | ~63 |
| **beam запускался только для «скользящих» головоломок** — `if sol is None and sliding`. Цилиндр и скрытые получали только A*, который без хорошей `V` почти ничего не решал. Сделали beam запасным вариантом для *всех* типов. | **63 → 70** |
| **Таблица BFS `max_depth=6`** — тестовые инстансы намешаны случайным блужданием 30–100 шагов (оптимальный путь 15–40). Таблица глубины 6 покрывала ~0% из них. Подняли до `max_depth=30`, `max_states=500_000`. | **70 → 82** |

Другие исправления корректности:
- Захардкоженные `CONTENT_NUM/CONTENT_EMPTY` вместо импорта из `gym.py` → неверно для скрытых головоломок.
- **Multiprocessing + PyTorch:** загрузка модели *до* `fork()` роняла воркеры. Исправление: грузим `V` внутри каждого воркера *после* fork.
- Компактные целочисленные ключи состояния вместо JSON в горячих циклах → ~1.5× быстрее (`7.14 мкс → 4.39 мкс` на состояние).

## Почему инженерия производительности была важна не меньше алгоритма

Это Python при **жёстком лимите в 25 минут**. *Качество* эвристики почти не меняло долю решённых пятнашек — а вот *скорость* генерации потомков меняла сильно:

| Изменение | Пропускная способность | Решено пятнашек |
|---|---|---|
| `ordered_children` через gym | ~40k потомков/с | 1/8 |
| быстрый swap списка + компактный ключ | ~114k потомков/с | **6/8** |

Та же эвристика, ~3× быстрее переходы → **6× больше решённых инстансов** за тот же бюджет. При лимите времени настоящая цель — *число состояний, исследуемых в секунду*.

## Итог

1. **Сначала точные решатели с учётом структуры** — GF(2) для «Лампочек» оптимален и мгновенен; ни одна обученная модель его не побьёт.
2. **Обученная `V(s)` — запасной вариант, а не ядро** — она бьёт неинформированный A*, но проигрывает точным решателям и часто beam-у.
3. **Универсальный beam search с мягкой оценкой несовпадения** — лучший «на все случаи» для неизвестных головоломок: разнообразие важнее острой, но хрупкой эвристики.
4. **Большая часть скора пришла от багов и подбора глубины/бюджета**, а не от архитектуры.
5. **При лимите по времени цель — пропускная способность** — быстрые нативные переходы решили в 6× больше, чем более умная эвристика.

**Непроверенная идея на будущее: table-guided beam / встреча посередине** — beam идёт вперёд ~12 шагов, обратная таблица покрывает ~20 от решения → суммарное покрытие ~32, без дополнительного обучения.